# Impact of socio-economic factors on real estate prices in New York City

## Introduction

Real estate prices in New York City are influenced by a complex interplay of geographic, economic, and social variables. This analysis shows the socio-economic status (SES) of the population and property market values. While location is always important, we can now use data to look deeper into each neighborhood.
By combining real estate transaction data with US Census demographics, this study moves beyond traditional property valuation. We aim to understand why certain areas have higher prices by exploring how variables such as income, professional employment, and commute act as the primary drivers of real estate pricing. We will analyze these trends across all five boroughs: Manhattan, The Bronx, Brooklyn, Queens, and Staten Island.

## Data integration and methodology
To achieve a comprehensive analysis this study merges two datasets:
1. NYC Sales Dataset 2016-2017. It containing transaction prices, property types, and geographical identifiers.
2. US Census Data 2013-2017. It providing demographic and socio-economic metrics at the census tract level.
The connection between these datasets is facilitated through the **Geographic Correspondence** engine allowing us to map census tracts to ZIP codes (ZCTAs) for a precise merge.

**Temporal Alignment (ACS 2017)** \
I use the **ACS 2017 5-year estimates** for the socio-economic part. Even though the property sales data covers a specific period September 2016 - September 2017, the government doesn't publish monthly population data for small neighborhoods. However neighborhood characteristics like income and professional status don't change drastically in a few months. Therefore the 2017 dataset provides a perfect basis for undestanding of the conditions at the time these properties were being sold.

## Analytical framework
The research follows a clear step-by-step process:
1. Data preprocessing \
Cleaning and merging datasets to ensure integrity and alignment. А key step here is Log transformation to handle the right-skewed nature of real estate prices:
$$ \ln(P) = \text{log\_sale\_price} $$

3. Exploratory Data Analysis (EDA) \
Using histograms and scatter plots to visualize trends and relationships. We also use the Pearson correlation coefficient (***r***) to measure the strength and direction of the linear relationship between socio-economic variables and property prices:
$$ r = \frac{\sum (x_i - \bar{x})(y_i - \bar{y})}{\sqrt{\sum (x_i - \bar{x})^2 \sum (y_i - \bar{y})^2}} $$

5. Statistical validation \
Applying Levene’s test to check for homogeneity of variance (homoscedasticity):
$$ H_0: \sigma_1^2 = \sigma_2^2 = \dots = \sigma_k^2 $$
We also use Q-Q plots to verify if our log-transformed data follows a normal distribution.

6. Hypothesis testing \
We use Two-Way ANOVA to see if the Borough and Tax group have a real impact on prices. This is measured by the F-statistic, which compares the differences between groups to the variation within them: Utilizing T-tests and two-way ANOVA to determine how different factors affect the price. 
$$ F = \frac{\text{Variance between boroughs}}{\text{Variance within boroughs}} $$
A high F-value will prove that location and building type are significant drivers of the NYC real estate market. We also use independent T-tests to compare specific pairs of boroughs. This helps us see if the price difference between two areas is statistically significant
county_to_borough = {
    'New York County': 'Manhattan',
    'Kings County': 'Brooklyn',
    'Queens County': 'Queens',
    'Bronx County': 'Bronx',
    'Richmond County': 'Staten Island'
}
7. Bayesian inference \
We use Bayes' theorem to calculate probabilities. For example, if we know a property is "Elite" (in the top 10% of the market), we calculate the probability that it is located in a specific borough:
$$ P(\text{Borough} | \text{Elite}) = \frac{P(\text{Elite} | \text{Borough}) \cdot P(\text{Borough})}{P(\text{Elite})} $$

8. Fourier аnalysis \
We use the Fourier transform to see if the housing market follows any waves or cycles (seasonality) over time:
$$ \hat{f}(\xi) = \int_{-\infty}^{\infty} f(x) e^{-2\pi i x \xi} dx $$


In [71]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re

## 1. Data cleaning of first dataset - NYC property sales

In [3]:
nyc_dataset = pd.read_csv("nyc-rolling-sales.csv")

In [4]:
nyc_dataset

,Unnamed: 0,BOROUGH,NEIGHBORHOOD,BUILDING CLASS CATEGORY,TAX CLASS AT PRESENT,BLOCK,LOT,EASE-MENT,BUILDING CLASS AT PRESENT,ADDRESS,...,RESIDENTIAL UNITS,COMMERCIAL UNITS,TOTAL UNITS,LAND SQUARE FEET,GROSS SQUARE FEET,YEAR BUILT,TAX CLASS AT TIME OF SALE,BUILDING CLASS AT TIME OF SALE,SALE PRICE,SALE DATE
0,4,1,ALPHABET CITY,07 RENTALS - WALKUP APARTMENTS,2A,392,6,,C2,153 AVENUE B,...,5,0,5,1633,6440,1900,2,C2,6625000,2017-07-19 00:00:00
1,5,1,ALPHABET CITY,07 RENTALS - WALKUP APARTMENTS,2,399,26,,C7,234 EAST 4TH STREET,...,28,3,31,4616,18690,1900,2,C7,-,2016-12-14 00:00:00
2,6,1,ALPHABET CITY,07 RENTALS - WALKUP APARTMENTS,2,399,39,,C7,197 EAST 3RD STREET,...,16,1,17,2212,7803,1900,2,C7,-,2016-12-09 00:00:00
3,7,1,ALPHABET CITY,07 RENTALS - WALKUP APARTMENTS,2B,402,21,,C4,154 EAST 7TH STREET,...,10,0,10,2272,6794,1913,2,C4,3936272,2016-09-23 00:00:00
4,8,1,ALPHABET CITY,07 RENTALS - WALKUP APARTMENTS,2A,404,55,,C2,301 EAST 10TH STREET,...,6,0,6,2369,4615,1900,2,C2,8000000,2016-11-17 00:00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
84543,8409,5,WOODROW,02 TWO FAMILY DWELLINGS,1,7349,34,,B9,37 QUAIL LANE,...,2,0,2,2400,2575,1998,1,B9,450000,2016-11-28 00:00:00
84544,8410,5,WOODROW,02 TWO FAMILY DWELLINGS,1,7349,78,,B9,32 PHEASANT LANE,...,2,0,2,2498,2377,1998,1,B9,550000,2017-04-21 00:00:00
84545,8411,5,WOODROW,02 TWO FAMILY DWELLINGS,1,7351,60,,B2,49 PITNEY AVENUE,...,2,0,2,4000,1496,1925,1,B2,460000,2017-07-05 00:00:00
84546,8412,5,WOODROW,22 STORE BUILDINGS,4,7100,28,,K6,2730 ARTHUR KILL ROAD,...,0,7,7,208033,64117,2001,4,K6,11693337,2016-12-21 00:00:00


In [5]:
nyc_dataset.head()

,Unnamed: 0,BOROUGH,NEIGHBORHOOD,BUILDING CLASS CATEGORY,TAX CLASS AT PRESENT,BLOCK,LOT,EASE-MENT,BUILDING CLASS AT PRESENT,ADDRESS,...,RESIDENTIAL UNITS,COMMERCIAL UNITS,TOTAL UNITS,LAND SQUARE FEET,GROSS SQUARE FEET,YEAR BUILT,TAX CLASS AT TIME OF SALE,BUILDING CLASS AT TIME OF SALE,SALE PRICE,SALE DATE
0,4,1,ALPHABET CITY,07 RENTALS - WALKUP APARTMENTS,2A,392,6,,C2,153 AVENUE B,...,5,0,5,1633,6440,1900,2,C2,6625000,2017-07-19 00:00:00
1,5,1,ALPHABET CITY,07 RENTALS - WALKUP APARTMENTS,2,399,26,,C7,234 EAST 4TH STREET,...,28,3,31,4616,18690,1900,2,C7,-,2016-12-14 00:00:00
2,6,1,ALPHABET CITY,07 RENTALS - WALKUP APARTMENTS,2,399,39,,C7,197 EAST 3RD STREET,...,16,1,17,2212,7803,1900,2,C7,-,2016-12-09 00:00:00
3,7,1,ALPHABET CITY,07 RENTALS - WALKUP APARTMENTS,2B,402,21,,C4,154 EAST 7TH STREET,...,10,0,10,2272,6794,1913,2,C4,3936272,2016-09-23 00:00:00
4,8,1,ALPHABET CITY,07 RENTALS - WALKUP APARTMENTS,2A,404,55,,C2,301 EAST 10TH STREET,...,6,0,6,2369,4615,1900,2,C2,8000000,2016-11-17 00:00:00


In [6]:
nyc_dataset.dtypes

Unnamed: 0                         int64
BOROUGH                            int64
NEIGHBORHOOD                      object
BUILDING CLASS CATEGORY           object
TAX CLASS AT PRESENT              object
BLOCK                              int64
LOT                                int64
EASE-MENT                         object
BUILDING CLASS AT PRESENT         object
ADDRESS                           object
APARTMENT NUMBER                  object
ZIP CODE                           int64
RESIDENTIAL UNITS                  int64
COMMERCIAL UNITS                   int64
TOTAL UNITS                        int64
LAND SQUARE FEET                  object
GROSS SQUARE FEET                 object
YEAR BUILT                         int64
TAX CLASS AT TIME OF SALE          int64
BUILDING CLASS AT TIME OF SALE    object
SALE PRICE                        object
SALE DATE                         object
dtype: object

In [7]:
nyc_dataset.shape

(84548, 22)

In [8]:
nyc_dataset.columns = nyc_dataset.columns.str.lower().str.replace(' ', '_')

In [9]:
nyc_dataset['unnamed:_0']

0           4
1           5
2           6
3           7
4           8
         ... 
84543    8409
84544    8410
84545    8411
84546    8412
84547    8413
Name: unnamed:_0, Length: 84548, dtype: int64

In [10]:
nyc_dataset.drop(columns = ['unnamed:_0'], inplace = True)

In [11]:
nyc_dataset.shape

(84548, 21)

In [12]:
nyc_dataset.isnull().sum()

borough                           0
neighborhood                      0
building_class_category           0
tax_class_at_present              0
block                             0
lot                               0
ease-ment                         0
building_class_at_present         0
address                           0
apartment_number                  0
zip_code                          0
residential_units                 0
commercial_units                  0
total_units                       0
land_square_feet                  0
gross_square_feet                 0
year_built                        0
tax_class_at_time_of_sale         0
building_class_at_time_of_sale    0
sale_price                        0
sale_date                         0
dtype: int64

Some cells maybe contain spaces or dashes ('-') instead of being empty. We need replace them with NaN values to reveal the count of an actual missing data.

In [13]:
nyc_dataset = nyc_dataset.apply(lambda x: x.str.strip() if x.dtype == "object" else x)

In [14]:
pd.set_option('future.no_silent_downcasting', True)

In [15]:
nyc_dataset.replace(['-', '', ' '], np.nan, inplace = True)

I set the ***future.no_silent_downcasting*** option to True. This ensures the code remains compatible with future versions of Pandas and prevents unnecessary warning messages when replacing values.

In [16]:
nyc_dataset = nyc_dataset.infer_objects(copy=False)

In [17]:
print(nyc_dataset.isnull().sum())

borough                               0
neighborhood                          0
building_class_category               0
tax_class_at_present                738
block                                 0
lot                                   0
ease-ment                         84548
building_class_at_present           738
address                               0
apartment_number                  65496
zip_code                              0
residential_units                     0
commercial_units                      0
total_units                           0
land_square_feet                  26252
gross_square_feet                 27612
year_built                            0
tax_class_at_time_of_sale             0
building_class_at_time_of_sale        0
sale_price                        14561
sale_date                             0
dtype: int64


The columns ***land_square_feet***, ***gross_square_feet***, and ***sale_price*** are currently stored as objects. We need to convert them to numeric types to perform mathematical operations, such as calculating averages or analyzing price trends.

In [18]:
num_cols = ['land_square_feet', 'gross_square_feet', 'sale_price']
for col in num_cols:
    nyc_dataset[col] = pd.to_numeric(nyc_dataset[col], errors = 'coerce')

In [19]:
nyc_dataset.dtypes

borough                             int64
neighborhood                       object
building_class_category            object
tax_class_at_present               object
block                               int64
lot                                 int64
ease-ment                         float64
building_class_at_present          object
address                            object
apartment_number                   object
zip_code                            int64
residential_units                   int64
commercial_units                    int64
total_units                         int64
land_square_feet                  float64
gross_square_feet                 float64
year_built                          int64
tax_class_at_time_of_sale           int64
building_class_at_time_of_sale     object
sale_price                        float64
sale_date                          object
dtype: object

In [20]:
nyc_dataset.sale_price

0         6625000.0
1               NaN
2               NaN
3         3936272.0
4         8000000.0
            ...    
84543      450000.0
84544      550000.0
84545      460000.0
84546    11693337.0
84547       69300.0
Name: sale_price, Length: 84548, dtype: float64

Deleting invalid price values -> ***sale_price < 10 000 & sale_price = 0***, because these values are not real sales. They are inherited estates or business sales. They don't have analytical value for our purpose. Also deleting NaN values, as they would distort with our calculations and could lead to biased or incorrect statistical results.

In [21]:
nyc_dataset.dropna(subset = ['sale_price'], inplace = True)

In [22]:
nyc_dataset.sale_price.isnull().sum()

np.int64(0)

In [23]:
(nyc_dataset.sale_price == 0).sum()

np.int64(10228)

In [24]:
(nyc_dataset.sale_price < 1000).sum()

np.int64(11306)

In [25]:
nyc_dataset = nyc_dataset[nyc_dataset.sale_price >= 10000].copy()

In [26]:
nyc_dataset.shape

(58465, 21)

There are residential and commerce buildings. For our analysis we need residential buildings. We need to remove other. For that purpose we can filter the data with ***tax_building_at_sale_price*** to remove business deals.
#### NYC Property Tax Classes
Class 1 - 1-3 Unit residential: 1- to 3-family homes, small condos, and vacant land zoned residential. These are generally assessed at a lower percentage of market value (6%). \
Class 2 - Apartments & co-ops: Rental apartment buildings, cooperatives, and condominiums with 4 or more units. \
Class 3 - Utility properties: Property owned by utility companies, including special franchise property. \
Class 4 - Commercial & industrial: Office buildings, factories, retail space, and warehouses.

In [27]:
nyc_dataset = nyc_dataset[nyc_dataset['tax_class_at_time_of_sale'].isin([1, 2])].copy()

Properties with more than three units are typically commercial investments rather than individual homes. To better analyze factors affecting the average person, we have restricted the dataset to properties with 3 units or fewer. This ensures that institutional investments do not skew our findings

In [28]:
nyc_dataset = nyc_dataset[nyc_dataset.total_units <= 3].copy()

In [29]:
len(nyc_dataset)

53665

In [30]:
pd.qcut(nyc_dataset['sale_price'], 10).value_counts().sort_index()

sale_price
(9999.999, 225000.0]       5395
(225000.0, 330000.0]       5400
(330000.0, 425000.0]       5364
(425000.0, 512000.0]       5311
(512000.0, 615000.0]       5393
(615000.0, 731912.4]       5336
(731912.4, 885000.0]       5369
(885000.0, 1168000.0]      5366
(1168000.0, 1825993.6]     5364
(1825993.6, 98525704.0]    5367
Name: count, dtype: int64

Refining the dataset to focus on individual residential sales. By selecting Tax Classes 1 and 2 and limiting the properties to 3 units or fewer, we exclude large-scale commercial investments and massive apartment complexes. This leaves us with 53,665 records representing the primary housing market. High-value outliers (up to $98M) are intentionally retained at this stage to observe the full diversity of the NYC luxury sector before final statistical normalization.

The column ease-ment contains no useful information, it is entirely empty, so we are removing it from the dataset.

In [31]:
nyc_dataset = nyc_dataset.drop(columns=['ease-ment']).copy()

The column apartment_number is missing more than 70% of its data. Since it doesn't provide significant value, we have decided to remove it.

In [32]:
nyc_dataset = nyc_dataset.drop(columns = ['apartment_number']).copy()

In [33]:
(nyc_dataset.land_square_feet == 0).sum()

np.int64(7799)

Filling missing values in column ***gross_square_feet*** using the median value for each specific neighborhood. The median is preferred over the mean because real estate data is often skewed by extreme outliers like very large or very small properties.This gives us a more realistic view of a typical property and prevents our analysis from being distorted.

In [34]:
nyc_dataset.land_square_feet = nyc_dataset.groupby('neighborhood')['land_square_feet'].transform(lambda x: x.fillna(x.median()))

In [35]:
nyc_dataset.land_square_feet.isnull().sum()

np.int64(3252)

After the first step, 3252 missing values remain. This occurs because some entire neighborhoods have no data for square footage, making it impossible to calculate a local median. In these cases, we use the median of the entire borough instead. This broader category allows us to fill the remaining gaps effectively.

In [36]:
land_square_medians = nyc_dataset.groupby('borough')['land_square_feet'].transform('median')

In [37]:
nyc_dataset.land_square_feet = nyc_dataset.land_square_feet.fillna(land_square_medians)

In [38]:
nyc_dataset.land_square_feet.isnull().sum()

np.int64(0)

In [39]:
(nyc_dataset['year_built'] == 0).sum()

np.int64(3657)

The ***year_built*** column has values of 0, which is impossible. I will replace these zeros with the median year of each neighborhood. This ensures every property has a realistic construction year based on its location.

In [40]:
nyc_dataset['year_built'] = nyc_dataset['year_built'].replace(0, np.nan)

In [41]:
nyc_dataset['year_built'] = nyc_dataset.groupby('neighborhood')['year_built'].transform(lambda x: x.fillna(x.median()))

In [42]:
(nyc_dataset['year_built'] == 0).sum()

np.int64(0)

Converting ***sale_date*** to datetime format and extracting the day of the week to analyze sales patterns.

In [44]:
nyc_dataset.sale_date = pd.to_datetime(nyc_dataset.sale_date, errors = 'coerce')

In [45]:
nyc_dataset['sale_date'] = nyc_dataset['sale_date'].dt.date

In [46]:
nyc_dataset['sale_date'] = pd.to_datetime(nyc_dataset['sale_date'])

In [47]:
nyc_dataset['sale_day_of_week'] = nyc_dataset['sale_date'].dt.day_name()

In [48]:
print(nyc_dataset[['sale_date', 'sale_day_of_week']].head())

    sale_date sale_day_of_week
13 2017-03-10           Friday
15 2017-06-09           Friday
16 2017-07-14           Friday
17 2017-03-16         Thursday
18 2016-09-01         Thursday


In [49]:
nyc_dataset['sale_day_of_week'].value_counts()

sale_day_of_week
Thursday     12665
Wednesday    11040
Friday       10734
Tuesday      10310
Monday        8777
Saturday        90
Sunday          49
Name: count, dtype: int64

We checked the sale dates to see which days are the busiest. Almost all sales happen on weekdays, with Thursday being the most popular day. Very few sales happen on weekends because banks and lawyers are closed. This confirms that our data is realistic.
A very small number of sales are recorded on weekends. This could be due to private contracts signed on those days or minor data entry errors. However, since it's less than 0.3%, it doesn't affect our overall analysis.

Standardizing text data in colomns ***neighborhood***, ***category***, and ***address***.

In [51]:
cols_to_fix = ['neighborhood', 'building_class_category', 'address']
for col in cols_to_fix:
    nyc_dataset[col] = nyc_dataset[col].str.strip().str.title()

In [52]:
nyc_dataset

,borough,neighborhood,building_class_category,tax_class_at_present,block,lot,building_class_at_present,address,zip_code,residential_units,commercial_units,total_units,land_square_feet,gross_square_feet,year_built,tax_class_at_time_of_sale,building_class_at_time_of_sale,sale_price,sale_date,sale_day_of_week
13,1,Alphabet City,09 Coops - Walkup Apartments,2,373,40,C6,"327 East 3 Street, 1C",10009,0,0,0,1761.5,NaN,1920.0,2,C6,499000.0,2017-03-10,Friday
15,1,Alphabet City,09 Coops - Walkup Apartments,2,373,40,C6,"327 East 3Rd Street, 5A",10009,0,0,0,1761.5,NaN,1920.0,2,C6,529500.0,2017-06-09,Friday
16,1,Alphabet City,09 Coops - Walkup Apartments,2,373,40,C6,"327 East 3 Street, 2E",10009,0,0,0,1761.5,NaN,1920.0,2,C6,423000.0,2017-07-14,Friday
17,1,Alphabet City,09 Coops - Walkup Apartments,2,373,46,C6,"317 East 3Rd Street, 12",10009,0,0,0,1761.5,NaN,1925.0,2,C6,501000.0,2017-03-16,Thursday
18,1,Alphabet City,09 Coops - Walkup Apartments,2,373,49,C6,"311 East 3Rd Street, 17",10009,0,0,0,1761.5,NaN,1920.0,2,C6,450000.0,2016-09-01,Thursday
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
84540,5,Woodrow,02 Two Family Dwellings,1,7316,93,B2,125 Darnell Lane,10309,2,0,2,3325.0,1300.0,1995.0,1,B2,509000.0,2016-10-31,Monday
84541,5,Woodrow,02 Two Family Dwellings,1,7317,126,B2,112 Robin Court,10309,2,0,2,11088.0,2160.0,1994.0,1,B2,648000.0,2016-12-07,Wednesday
84543,5,Woodrow,02 Two Family Dwellings,1,7349,34,B9,37 Quail Lane,10309,2,0,2,2400.0,2575.0,1998.0,1,B9,450000.0,2016-11-28,Monday
84544,5,Woodrow,02 Two Family Dwellings,1,7349,78,B9,32 Pheasant Lane,10309,2,0,2,2498.0,2377.0,1998.0,1,B9,550000.0,2017-04-21,Friday


In [53]:
nyc_dataset.duplicated().sum()

np.int64(73)

In [54]:
nyc_dataset = nyc_dataset.drop_duplicates()

In [55]:
nyc_dataset = nyc_dataset.copy()

We need to drop the ***'at present'*** tax and building class columns because we are only interested in the information at the time of the sale. This keeps the dataset focused on the historical transaction data.

In [56]:
nyc_dataset.drop(columns = ['building_class_at_present', 'tax_class_at_present'], inplace = True)

### 2. Data cleaning of second dataset - NYC census tract

We will use the **ACS 2017 5-year estimates** (Version 3 of the Kaggle dataset). Although the general description mentions 2015, the specific file acs2017_census_tract_data.csv contains the updated data for the 2013-2017 period. \
The original dataset contains census data for the entire United States. We will filter it to include only **New York City** by selecting the five specific counties that make up the city - **New York (Manhattan)**, **Kings (Brooklyn)**, **Queens**, **Bronx**, and **Richmond (Staten Island)**. This allows us to focus our socio-economic analysis exclusively on the NYC area. \

This dataset provides a detailed profile of each neighborhood through several key categories:
1. Demographics - total population and ethnic diversity (white, black, hispanic, asian, etc.) 
2. Income and wealth - median household income, income per capita, and poverty rates. 
3. Employment/unemployment rates and occupation types (professional, service, construction, etc.).
4. Commuting - common transport methods (public, transit, walking, driving) and average commute times.

A Census Tract ID is a unique identification number assigned to small geographic areas by the Census Bureau. It acts as a "common key" that allows us to link property sales to the specific socio-economic data, like income and poverty, of the area where the building is located.


In [57]:
#Census data for the entire United States
us_census_data = pd.read_csv('acs2017_census_tract_data.csv')

In [58]:
us_census_data

,TractId,State,County,TotalPop,Men,Women,Hispanic,White,Black,Native,...,Walk,OtherTransp,WorkAtHome,MeanCommute,Employed,PrivateWork,PublicWork,SelfEmployed,FamilyWork,Unemployment
0,1001020100,Alabama,Autauga County,1845,899,946,2.4,86.3,5.2,0.0,...,0.5,0.0,2.1,24.5,881,74.2,21.2,4.5,0.0,4.6
1,1001020200,Alabama,Autauga County,2172,1167,1005,1.1,41.6,54.5,0.0,...,0.0,0.5,0.0,22.2,852,75.9,15.0,9.0,0.0,3.4
2,1001020300,Alabama,Autauga County,3385,1533,1852,8.0,61.4,26.5,0.6,...,1.0,0.8,1.5,23.1,1482,73.3,21.1,4.8,0.7,4.7
3,1001020400,Alabama,Autauga County,4267,2001,2266,9.6,80.3,7.1,0.5,...,1.5,2.9,2.1,25.9,1849,75.8,19.7,4.5,0.0,6.1
4,1001020500,Alabama,Autauga County,9965,5054,4911,0.9,77.5,16.4,0.0,...,0.8,0.3,0.7,21.0,4787,71.4,24.1,4.5,0.0,2.3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
73996,72153750501,Puerto Rico,Yauco Municipio,6011,3035,2976,99.7,0.3,0.0,0.0,...,0.5,0.0,3.6,26.9,1576,59.2,33.8,7.0,0.0,20.8
73997,72153750502,Puerto Rico,Yauco Municipio,2342,959,1383,99.1,0.9,0.0,0.0,...,0.0,0.0,1.3,25.3,666,58.4,35.4,6.2,0.0,26.3
73998,72153750503,Puerto Rico,Yauco Municipio,2218,1001,1217,99.5,0.2,0.0,0.0,...,3.4,0.0,3.4,23.5,560,57.5,34.5,8.0,0.0,23.0
73999,72153750601,Puerto Rico,Yauco Municipio,4380,1964,2416,100.0,0.0,0.0,0.0,...,0.0,0.0,0.0,24.1,1062,67.7,30.4,1.9,0.0,29.5


In [59]:
#Filter the data to include only NYC counties
nyc_counties = ['New York County', 'Kings County', 'Queens County', 'Bronx County', 'Richmond County']

In [62]:
nyc_census = us_census_data[(us_census_data['State'] == 'New York') & 
                         (us_census_data['County'].isin(nyc_counties))].copy()

In [63]:
county_to_borough = {
    'New York County': 'Manhattan',
    'Kings County': 'Brooklyn',
    'Queens County': 'Queens',
    'Bronx County': 'Bronx',
    'Richmond County': 'Staten Island'
}

In [64]:
#Add a new column 'borough' for easier recognition
nyc_census['borough'] = nyc_census['County'].map(county_to_borough)

To make the data more readable we created a mapping dictionary to convert official county names (like Kings County) into their well-known borough names (like Brooklyn). Finally, we added a new borough column to the dataset for easier identification during analysis.

In [65]:
nyc_census

,TractId,State,County,TotalPop,Men,Women,Hispanic,White,Black,Native,...,OtherTransp,WorkAtHome,MeanCommute,Employed,PrivateWork,PublicWork,SelfEmployed,FamilyWork,Unemployment,borough
43283,36005000100,New York,Bronx County,7411,6857,554,32.0,7.2,57.6,0.3,...,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,Bronx
43284,36005000200,New York,Bronx County,5058,2502,2556,77.7,0.8,18.4,0.0,...,0.9,0.5,41.8,1858,75.1,18.9,5.9,0.0,12.4,Bronx
43285,36005000400,New York,Bronx County,5944,2984,2960,66.4,3.4,28.6,0.0,...,0.6,1.9,47.7,2917,67.2,28.7,4.0,0.0,7.4,Bronx
43286,36005001600,New York,Bronx County,6115,2351,3764,63.3,3.7,31.1,0.0,...,0.0,3.9,38.6,2120,71.0,26.6,2.5,0.0,9.1,Bronx
43287,36005001900,New York,Bronx County,2817,1327,1490,53.4,7.7,33.1,0.0,...,1.9,1.9,44.1,1290,75.3,16.7,8.0,0.0,13.4,Bronx
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
47144,36085030302,New York,Richmond County,6560,3459,3101,38.5,33.4,11.9,0.0,...,0.0,1.3,48.6,2721,73.6,24.2,2.1,0.0,4.0,Staten Island
47145,36085031901,New York,Richmond County,2737,1060,1677,32.9,6.7,57.1,0.0,...,0.0,0.0,45.0,683,77.0,17.9,5.1,0.0,8.1,Staten Island
47146,36085031902,New York,Richmond County,5140,2409,2731,35.1,8.7,51.8,0.0,...,0.0,0.5,49.8,1714,82.3,15.3,2.5,0.0,4.6,Staten Island
47147,36085032300,New York,Richmond County,1136,578,558,29.3,22.7,45.7,0.0,...,0.0,0.0,47.3,544,80.9,15.6,3.5,0.0,4.9,Staten Island


In [67]:
nyc_census.drop(columns=['State'], inplace=True)

Since we have already filtered the data for New York City, the **State** column is no longer necessary. I drop it to keep the dataset clean.

In [68]:
nyc_census.dtypes

TractId               int64
County               object
TotalPop              int64
Men                   int64
Women                 int64
Hispanic            float64
White               float64
Black               float64
Native              float64
Asian               float64
Pacific             float64
VotingAgeCitizen      int64
Income              float64
IncomeErr           float64
IncomePerCap        float64
IncomePerCapErr     float64
Poverty             float64
ChildPoverty        float64
Professional        float64
Service             float64
Office              float64
Construction        float64
Production          float64
Drive               float64
Carpool             float64
Transit             float64
Walk                float64
OtherTransp         float64
WorkAtHome          float64
MeanCommute         float64
Employed              int64
PrivateWork         float64
PublicWork          float64
SelfEmployed        float64
FamilyWork          float64
Unemployment        

In [72]:
nyc_census.columns = [re.sub(r'(?<!^)(?=[A-Z])', '_', col).lower() for col in nyc_census.columns]

The first dataset is already in this format. By standardizing the second dataset to match, we ensure that both tables use a consistent snake_case naming convention. This makes it much easier to perform the merge and reference columns.

In [73]:
nyc_census.columns

Index(['tract_id', 'county', 'total_pop', 'men', 'women', 'hispanic', 'white',
       'black', 'native', 'asian', 'pacific', 'voting_age_citizen', 'income',
       'income_err', 'income_per_cap', 'income_per_cap_err', 'poverty',
       'child_poverty', 'professional', 'service', 'office', 'construction',
       'production', 'drive', 'carpool', 'transit', 'walk', 'other_transp',
       'work_at_home', 'mean_commute', 'employed', 'private_work',
       'public_work', 'self_employed', 'family_work', 'unemployment',
       'borough'],
      dtype='object')

In [74]:
nyc_census.isnull().sum()

tract_id               0
county                 0
total_pop              0
men                    0
women                  0
hispanic              40
white                 40
black                 40
native                40
asian                 40
pacific               40
voting_age_citizen     0
income                66
income_err            66
income_per_cap        48
income_per_cap_err    48
poverty               43
child_poverty         60
professional          45
service               45
office                45
construction          45
production            45
drive                 45
carpool               45
transit               45
walk                  45
other_transp          45
work_at_home          45
mean_commute          62
employed               0
private_work          45
public_work           45
self_employed         45
family_work           45
unemployment          45
borough                0
dtype: int64

In [75]:
(nyc_census.isnull().mean() * 100).round(2)

tract_id              0.00
county                0.00
total_pop             0.00
men                   0.00
women                 0.00
hispanic              1.85
white                 1.85
black                 1.85
native                1.85
asian                 1.85
pacific               1.85
voting_age_citizen    0.00
income                3.05
income_err            3.05
income_per_cap        2.22
income_per_cap_err    2.22
poverty               1.98
child_poverty         2.77
professional          2.08
service               2.08
office                2.08
construction          2.08
production            2.08
drive                 2.08
carpool               2.08
transit               2.08
walk                  2.08
other_transp          2.08
work_at_home          2.08
mean_commute          2.86
employed              0.00
private_work          2.08
public_work           2.08
self_employed         2.08
family_work           2.08
unemployment          2.08
borough               0.00
d

In [76]:
nyc_census.dropna(inplace = True)

We checked for missing values and found that they represented a very small percentage of the dataset. I decided to drop these rows to ensure that our analysis is based only on complete and reliable information. 

In [77]:
nyc_census.isnull().sum()

tract_id              0
county                0
total_pop             0
men                   0
women                 0
hispanic              0
white                 0
black                 0
native                0
asian                 0
pacific               0
voting_age_citizen    0
income                0
income_err            0
income_per_cap        0
income_per_cap_err    0
poverty               0
child_poverty         0
professional          0
service               0
office                0
construction          0
production            0
drive                 0
carpool               0
transit               0
walk                  0
other_transp          0
work_at_home          0
mean_commute          0
employed              0
private_work          0
public_work           0
self_employed         0
family_work           0
unemployment          0
borough               0
dtype: int64

In [79]:
nyc_census.duplicated().sum()

np.int64(0)